In [7]:
import os

os.makedirs("files/maestro-v2.0.0/train", exist_ok=True)
os.makedirs("files/maestro-v2.0.0/val", exist_ok=True)
os.makedirs("files/maestro-v2.0.0/test", exist_ok=True)

In [9]:
import json
import pickle
from processor import encode_midi

file="files/maestro-v2.0.0/maestro-v2.0.0.json"

with open(file,"r") as fb:
    # 加载 JSON 文件
    maestro_json=json.load(fb)
# 遍历训练数据中的所有文件
for x in maestro_json:
    mid=rf'files/maestro-v2.0.0/{x["midi_filename"]}'
    # 根据 JSON 文件中的指令，将文件放入训练、验证或测试子文件夹
    split_type = x["split"]
    f_name = mid.split("/")[-1] + ".pickle"
    if(split_type == "train"):
        o_file = rf'files/maestro-v2.0.0/train/{f_name}'
    elif(split_type == "validation"):
        o_file = rf'files/maestro-v2.0.0/val/{f_name}'
    elif(split_type == "test"):
        o_file = rf'files/maestro-v2.0.0/test/{f_name}'
    prepped = encode_midi(mid)
    with open(o_file,"wb") as f:
        pickle.dump(prepped, f)

In [10]:
train_size=len(os.listdir('files/maestro-v2.0.0/train'))
print(f"there are {train_size} files in the train set")
val_size=len(os.listdir('files/maestro-v2.0.0/val'))
print(f"there are {val_size} files in the validation set")
test_size=len(os.listdir('files/maestro-v2.0.0/test'))
print(f"there are {test_size} files in the test set")

there are 967 files in the train set
there are 137 files in the validation set
there are 178 files in the test set


In [11]:
import pickle
from processor import encode_midi
import pretty_midi
from processor import (_control_preprocess,
    _note_preprocess,_divide_note,
    _make_time_sift_events,_snote2events)

# 从训练集中选择一个 MIDI 文件
file='MIDI-Unprocessed_Chamber1_MID--AUDIO_07_R3_2018_wav--2'
name=rf'files/maestro-v2.0.0/2018/{file}.midi'

events=[]
notes=[]

# convert song to an easily-manipulable format
song=pretty_midi.PrettyMIDI(name)
for inst in song.instruments:
    inst_notes=inst.notes
    ctrls=_control_preprocess([ctrl for ctrl in 
       inst.control_changes if ctrl.number == 64])
    # 从音乐中提取音乐事件
    notes += _note_preprocess(ctrls, inst_notes)
# 将所有音乐事件放入列表 dnotes 中
dnotes = _divide_note(notes)    
dnotes.sort(key=lambda x: x.time)    
for i in range(5):
    print(dnotes[i])   

<[SNote] time: 1.0325520833333333 type: note_on, value: 74, velocity: 86>
<[SNote] time: 1.0442708333333333 type: note_on, value: 38, velocity: 77>
<[SNote] time: 1.2265625 type: note_off, value: 74, velocity: None>
<[SNote] time: 1.2395833333333333 type: note_on, value: 73, velocity: 69>
<[SNote] time: 1.2408854166666665 type: note_on, value: 37, velocity: 64>


In [12]:
cur_time = 0
cur_vel = 0
for snote in dnotes:
    # 对时间进行离散化，以减少唯一事件的数量
    events += _make_time_sift_events(prev_time=cur_time,
                                     post_time=snote.time)
    # 将音符转换为事件
    events += _snote2events(snote=snote, prev_vel=cur_vel)
    cur_time = snote.time
    cur_vel = snote.velocity    
indexes=[e.to_int() for e in events]   
# 打印出前 15 个事件
for i in range(15):
    print(events[i])

<Event type: time_shift, value: 99>
<Event type: time_shift, value: 2>
<Event type: velocity, value: 21>
<Event type: note_on, value: 74>
<Event type: time_shift, value: 0>
<Event type: velocity, value: 19>
<Event type: note_on, value: 38>
<Event type: time_shift, value: 17>
<Event type: note_off, value: 74>
<Event type: time_shift, value: 0>
<Event type: velocity, value: 17>
<Event type: note_on, value: 73>
<Event type: velocity, value: 16>
<Event type: note_on, value: 37>
<Event type: time_shift, value: 0>


In [13]:
import torch,os,pickle

max_seq=2048
def create_xys(folder):  
    files=[os.path.join(folder,f) for f in os.listdir(folder)]
    xys=[]
    for f in files:
        with open(f,"rb") as fb:
            music=pickle.load(fb)
        music=torch.LongTensor(music)
        # 创建 (x, y) 序列，序列长度为 2,048 个索引，并将索引 399 设置为填充索引
        x=torch.full((max_seq,),389, dtype=torch.long)
        y=torch.full((max_seq,),389, dtype=torch.long)
        length=len(music)
        if length<=max_seq:
            print(length)
            # 使用最多 2,048 个索引的序列作为输入
            x[:length]=music
            # 将窗口向右滑动一个索引，并将其作为输出
            y[:length-1]=music[1:]
            # 将结束索引设置为 388
            y[length-1]=388    
        else:
            x=music[:max_seq]
            y=music[1:max_seq+1]   
        xys.append((x,y))
    return xys

In [14]:
trainfolder='files/maestro-v2.0.0/train'
train=create_xys(trainfolder)

1771
5
15
586
1643


In [15]:
valfolder='files/maestro-v2.0.0/val'
testfolder='files/maestro-v2.0.0/test'
print("processing the validation set")
val=create_xys(valfolder)
print("processing the test set")
test=create_xys(testfolder)

processing the validation set
processing the test set
1837


In [16]:
val1, _ = val[0]
print(val1.shape)
print(val1)

torch.Size([2048])
tensor([350, 364,  52,  ...,  67, 301, 370])


In [17]:
from processor import decode_midi

file_path="files/val1.midi"
decode_midi(val1.cpu().numpy(), file_path=file_path)

In [18]:
train1, _ = train[0]
file_path="files/train1.midi"
decode_midi(train1.cpu().numpy(), file_path=file_path)

In [19]:
from torch.utils.data import DataLoader

batch_size=2
trainloader=DataLoader(train,batch_size=batch_size,
                       shuffle=True)

In [20]:
from torch import nn
class Config():
    def __init__(self):
        self.n_layer = 6
        self.n_head = 8
        self.n_embd = 512
        self.vocab_size = 390
        self.block_size = 2048 
        self.embd_pdrop = 0.1
        self.resid_pdrop = 0.1
        self.attn_pdrop = 0.1
        
config=Config()
device="cuda" if torch.cuda.is_available() else "cpu"

In [21]:
from util import Model

model=Model(config)
model.to(device)
num=sum(p.numel() for p in model.transformer.parameters())
print("number of parameters: %.2fM" % (num/1e6,))
print(model)

number of parameters: 20.16M
Model(
  (transformer): ModuleDict(
    (wte): Embedding(390, 512)
    (wpe): Embedding(2048, 512)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=True)
          (c_proj): Linear(in_features=512, out_features=512, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=512, out_features=2048, bias=True)
          (c_proj): Linear(in_features=2048, out_features=512, bias=True)
          (act): GELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)


In [22]:
lr=0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=lr) 
loss_func=torch.nn.CrossEntropyLoss(ignore_index=389)

In [23]:
model.train()  
for i in range(1,101):
    tloss = 0.
    # 遍历所有训练数据批次
    for idx, (x,y) in enumerate(trainloader):
        x,y=x.to(device),y.to(device)
        output = model(x)
        # 将模型预测与实际输出进行比较
        loss=loss_func(output.view(-1,output.size(-1)),
                           y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        # 将梯度范数裁剪为 1
        nn.utils.clip_grad_norm_(model.parameters(),1)
        # 调整模型参数
        optimizer.step()
        tloss += loss.item()
    print(f'epoch {i} loss {tloss/(idx+1)}') 
# 保存训练完成的模型
torch.save(model.state_dict(),f'files/musicTrans.pth') 

epoch 1 loss 4.241516945283275
epoch 2 loss 3.9276230877096
epoch 3 loss 3.801712766166561
epoch 4 loss 3.715248024168093
epoch 5 loss 3.6334972179625646
epoch 6 loss 3.545294762150315
epoch 7 loss 3.465260861826337
epoch 8 loss 3.39053257585557
epoch 9 loss 3.3297733063540185
epoch 10 loss 3.269783865321766
epoch 11 loss 3.218120674949047
epoch 12 loss 3.1709251645182777
epoch 13 loss 3.12261365169336
epoch 14 loss 3.0764115777882663
epoch 15 loss 3.0305666091028325
epoch 16 loss 2.9890737060672983
epoch 17 loss 2.943609577565154
epoch 18 loss 2.8974793819356557
epoch 19 loss 2.848926989992788
epoch 20 loss 2.798955806030715
epoch 21 loss 2.7500410971562723
epoch 22 loss 2.7005357348229273
epoch 23 loss 2.6513171033425764
epoch 24 loss 2.6050473027978063
epoch 25 loss 2.5603588757928737
epoch 26 loss 2.517369283624917
epoch 27 loss 2.476511596648161
epoch 28 loss 2.433726055809289
epoch 29 loss 2.3924425276350383
epoch 30 loss 2.3553493106660763
epoch 31 loss 2.316397212753611
epoch 3

In [25]:
from processor import decode_midi

prompt, _  = test[42]
prompt = prompt.to(device)
len_prompt=250

file_path = "files/prompt.midi"
decode_midi(prompt[:len_prompt].cpu().numpy(),
            file_path=file_path)

In [26]:
prompt, _  = test[1]
prompt = prompt.to(device)
len_prompt=250
file_path = "files/prompt2.midi"
decode_midi(prompt[:len_prompt].cpu().numpy(),
            file_path=file_path)

In [ ]:
softmax=torch.nn.Softmax(dim=-1)
def sample(prompt,seq_length=1000,temperature=1):
    gen_seq=torch.full((1,seq_length),389,dtype=torch.long).to(device)
    idx=len(prompt)
    gen_seq[..., :idx]=prompt.type(torch.long).to(device)
    # 生成新的索引，直到序列达到指定长度
    while(idx < seq_length):
        # 将预测结果除以 temperature 参数，然后对 logits 应用 softmax 函数
        y=softmax(model(gen_seq[..., :idx])/temperature)[...,:388]
        probs=y[:, idx-1, :]
        distrib=torch.distributions.categorical.Categorical(probs=probs)
        # 从预测的概率分布中采样，以生成新的索引
        next_token=distrib.sample()
        gen_seq[:, idx]=next_token
        idx+=1
    # 输出整个序列
    return gen_seq[:, :idx]

In [30]:
model.load_state_dict(torch.load("files/musicTrans.pth", weights_only=False))
model.eval()

Model(
  (transformer): ModuleDict(
    (wte): Embedding(390, 512)
    (wpe): Embedding(2048, 512)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=True)
          (c_proj): Linear(in_features=512, out_features=512, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=512, out_features=2048, bias=True)
          (c_proj): Linear(in_features=2048, out_features=512, bias=True)
          (act): GELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_fe

In [31]:
from processor import encode_midi

file_path = "files/prompt.midi"
prompt = torch.tensor(encode_midi(file_path))
generated_music=sample(prompt, seq_length=1000)

In [32]:
music_data = generated_music[0].cpu().numpy()
file_path = 'files/musicTrans.midi'
decode_midi(music_data, file_path=file_path)

info removed pitch: 73
info removed pitch: 60
info removed pitch: 80
info removed pitch: 39


In [33]:
file_path = "files/prompt2.midi"
prompt = torch.tensor(encode_midi(file_path))
generated_music=sample(prompt, seq_length=1200,temperature=1)
music_data = generated_music[0].cpu().numpy()
file_path = 'files/musicTrans2.midi'
decode_midi(music_data, file_path=file_path)

info removed pitch: 74


In [34]:
file_path = "files/prompt.midi"
prompt = torch.tensor(encode_midi(file_path))
generated_music=sample(prompt, seq_length=1000,temperature=1.5)
music_data = generated_music[0].cpu().numpy()
file_path = 'files/musicHiTemp.midi'
decode_midi(music_data, file_path=file_path)

info removed pitch: 80
info removed pitch: 55
info removed pitch: 87
info removed pitch: 76
info removed pitch: 70
info removed pitch: 88
info removed pitch: 76
info removed pitch: 33


In [35]:
file_path = "files/prompt.midi"
prompt = torch.tensor(encode_midi(file_path))
generated_music=sample(prompt, seq_length=1000,temperature=0.7)
music_data = generated_music[0].cpu().numpy()
file_path = 'files/musicLowTemp.midi'
decode_midi(music_data, file_path=file_path)

info removed pitch: 36
info removed pitch: 71
